In [12]:
import vcfpy
import pandas as pd
import requests
import json

In [13]:
vcf_reader = vcfpy.Reader(open('challenge_data.vcf', 'r'))

In [14]:
record = next(vcf_reader)

chrom = record.CHROM
pos = record.POS
ref = record.REF
alt = record.ALT[0] # ",".join(str(_alt) for _alt in record.ALT)

# 1. "DP": Depth of sequence coverage at site of variation
depth = record.INFO.get('DP', None)
allelic_depth = record.INFO.get('AD', None)

print(f"""CHROMOSOME: {chrom}
POSITION: {pos}
REFERENCE: {ref}
ALTERNATE: {alt}
DEPTH: {depth}
ALLELIC DEPTH: {allelic_depth}
""")

CHROMOSOME: 1
POSITION: 931393
REFERENCE: G
ALTERNATE: Substitution(type_='SNV', value='T')
DEPTH: 4124
ALLELIC DEPTH: None



It appears the VCF file is missing the allelic depth feature. This will prove problematic for determining a few of the requested annotation values.

Fortunately, the file contains the total depth and allele frequency. So we can compute the number of reads supporting the variant

In [15]:
# 2. Number of reads supporting the variant
allele_frequency = record.INFO.get('AF', None)

ref_depth = record.INFO.get('RO', None)
alt_depth = record.INFO.get('AO', None)

print(f"""CHROMOSOME: {chrom}
POSITION: {pos}
REFERENCE: {ref}
ALTERNATE: {alt}
DEPTH: {depth}
ALLELE FREQ: {allele_frequency}
REF COUNT: {ref_depth}
ALT COUNT: {alt_depth}
""")

CHROMOSOME: 1
POSITION: 931393
REFERENCE: G
ALTERNATE: Substitution(type_='SNV', value='T')
DEPTH: 4124
ALLELE FREQ: [0.0]
REF COUNT: 4029
ALT COUNT: [95]



In [16]:
# 3. Percentage of reads supporting the variant
#       versus those supporting reference reads
total_depth = ref_depth + alt_depth[0]
pct_variant = (alt_depth[0] / total_depth) * 100 if total_depth > 0 else 0

print(f"""REF DEPTH: {ref_depth}
ALT DEPTH: {alt_depth}
TOTAL DEPTH: {total_depth}
% VARIANT: {pct_variant}
""")

REF DEPTH: 4029
ALT DEPTH: [95]
TOTAL DEPTH: 4124
% VARIANT: 2.303588748787585



In [17]:
# 4. Query Ensemble VEP API
vep_params = {
    # 'region': f'{chrom}:{pos}:{pos}/{alt.value}',
    # "allele": f"{ref}/{alt.value}",
    # 'variants': f'{chrom} {pos} {ref} {alt}',
    'content-type': 'application/json'
}
# vep_payload = {
#     'variants': [f'{chrom} {pos} {ref} {alt}']
# }
VEP_API_URL = f"https://grch37.rest.ensembl.org/vep/human/region/{chrom}:{pos}/{alt.value}"

try:
    response = requests.get(VEP_API_URL, params=vep_params)
    # response = requests.post(VEP_API_URL, headers=vep_params, data=json.dumps(vep_payload))
    print(response.json())
except Exception as e:
    print(e)

[{'allele_string': 'G/T', 'assembly_name': 'GRCh37', 'input': '1 931393 931393 G/T 1', 'start': 931393, 'end': 931393, 'id': '1_931393_G/T', 'seq_region_name': '1', 'most_severe_consequence': 'non_coding_transcript_exon_variant', 'transcript_consequences': [{'gene_symbol_source': 'HGNC', 'strand': -1, 'variant_allele': 'T', 'gene_symbol': 'HES4', 'hgnc_id': 24149, 'distance': 2951, 'biotype': 'protein_coding', 'consequence_terms': ['downstream_gene_variant'], 'gene_id': 'ENSG00000188290', 'impact': 'MODIFIER', 'transcript_id': 'ENST00000304952'}, {'consequence_terms': ['downstream_gene_variant'], 'gene_id': 'ENSG00000188290', 'biotype': 'protein_coding', 'transcript_id': 'ENST00000428771', 'impact': 'MODIFIER', 'variant_allele': 'T', 'strand': -1, 'gene_symbol_source': 'HGNC', 'distance': 2949, 'gene_symbol': 'HES4', 'hgnc_id': 24149}, {'biotype': 'retained_intron', 'gene_id': 'ENSG00000188290', 'consequence_terms': ['downstream_gene_variant'], 'impact': 'MODIFIER', 'transcript_id': 'E

In [18]:
vep_data = response.json()
gene = vep_data[0]['transcript_consequences'][0]['gene_symbol']
effect_type = vep_data[0]['transcript_consequences'][0]['consequence_terms'][0]
most_severe_consequence = vep_data[0]['most_severe_consequence']

print(f"""GENE: {gene}
EFFECT TYPE: {effect_type}
MOST SEVERE CONSEQUENCE: {most_severe_consequence}
""")

GENE: HES4
EFFECT TYPE: downstream_gene_variant
MOST SEVERE CONSEQUENCE: non_coding_transcript_exon_variant

